In [3]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd

@dataclass(frozen=True)
class EmulatorConfig:
    seed: int = 42
    start_timestamp: str = "2026-01-01 00:00:00"
    duration_hours: int = 120
    interval_minutes: int = 1
    nominal_hours_end: float = 75.0
    ood_hours_start: float = 76.0
    train_cutoff_hours: float = 80.0
    drift_rate_kpa_per_hour: float = -0.075  # Strong transducer drift
    cavitation_threshold_kpa: float = -7.0
    output_name: str = "pump_telemetry.csv"

class PumpTelemetryEmulator:
    def __init__(self, config: EmulatorConfig | None = None) -> None:
        self.config = config or EmulatorConfig()
        self.rng = np.random.default_rng(self.config.seed)

    def _time_axis(self) -> np.ndarray:
        n_rows = int(self.config.duration_hours * 60 / self.config.interval_minutes)
        return np.arange(n_rows, dtype=float) * (self.config.interval_minutes / 60.0)

    def generate(self, output_path: Path) -> pd.DataFrame:
        time_hours = self._time_axis()
        n_samples = time_hours.size
        
        start = pd.Timestamp(self.config.start_timestamp)
        timestamps = pd.DatetimeIndex(start + pd.to_timedelta(time_hours, unit="h"))

        # 1. Vibration: Base noise + structural cavitation crackle at the end
        raw_vibration = 0.12 + 0.04 * np.sin(2 * np.pi * time_hours / 12.0) + self.rng.normal(0.0, 0.08, size=n_samples)
        
        # 2. Pressure: True physical performance + dynamic sensor calibration drift
        true_pressure = 120.0 + 2.5 * np.sin(2 * np.pi * time_hours / 24.0)
        drift_signal = self.config.drift_rate_kpa_per_hour * time_hours
        drifted_pressure = true_pressure + drift_signal + self.rng.normal(0.0, 0.2, size=n_samples)

        # 3. Motor Temperature: Steady thermal baseline
        motor_temp = 55.0 + 4.0 * np.sin(2 * np.pi * time_hours / 8.0) + self.rng.normal(0.0, 0.3, size=n_samples)

        # 4. Target Variable: Pump Hydraulic Efficiency
        pump_efficiency = 0.85 - 0.03 * (time_hours / self.config.duration_hours)
        
        # Introduce cavitation degradation phase
        fail_mask = time_hours >= self.config.ood_hours_start
        if fail_mask.any():
            fail_hours = time_hours[fail_mask] - self.config.ood_hours_start
            # Severe fluid breakdown drop
            pump_efficiency[fail_mask] -= 0.45 * (fail_hours / (self.config.duration_hours - self.config.ood_hours_start)) ** 1.4
            # Add violent structural vibration shocks
            raw_vibration[fail_mask] += self.rng.uniform(0.3, 1.2, size=fail_mask.sum())
            motor_temp[fail_mask] += 8.0 * (fail_hours / (self.config.duration_hours - self.config.ood_hours_start))

        pump_efficiency = np.clip(pump_efficiency + self.rng.normal(0.0, 0.005, size=n_samples), 0.10, 0.95)

        df = pd.DataFrame({
            "timestamp": timestamps,
            "time_hours": time_hours,
            "raw_vibration": raw_vibration,
            "drifted_pressure": drifted_pressure,
            "motor_temp": motor_temp,
            "pump_efficiency": pump_efficiency,
            "suggested_split": np.where(time_hours <= self.config.train_cutoff_hours, "train", "test")
        })

        # Inject real-world transmission dropouts (NaN rows)
        dropout_mask = self.rng.random(size=n_samples) < 0.03
        df.loc[dropout_mask, ["raw_vibration", "drifted_pressure", "motor_temp"]] = np.nan

        df.to_csv(output_path, index=False)
        print(f"Generated clean framework telemetry at: {output_path}")
        return df

if __name__ == "__main__":
    PumpTelemetryEmulator().generate(Path("pump_telemetry3.csv"))

Generated clean framework telemetry at: pump_telemetry3.csv
